# Загрузка данных

In [ ]:
# from google.colab import userdata
# import json
# import os

# KAGGLE_JSON_PATH = os.path.expanduser("~/.config/kaggle/kaggle.json")

# kaggle_dir = os.path.dirname(KAGGLE_JSON_PATH)
# os.makedirs(kaggle_dir, exist_ok=True)

# kaggle_json = json.loads(userdata.get("kaggle_json"))

# with open(KAGGLE_JSON_PATH, "w") as f:
#     json.dump(kaggle_json, f)

# !chmod 600 ~/.config/kaggle/kaggle.json

# !kaggle competitions download -c critical-temperature-of-superconductors
# !unzip /content/critical-temperature-of-superconductors.zip

  0% 0.00/8.64M [00:00<?, ?B/s]
100% 8.64M/8.64M [00:00<00:00, 492MB/s]
Archive:  /content/critical-temperature-of-superconductors.zip
  inflating: Samsung_Baseline____.ipynb  
  inflating: formula_test.csv        
  inflating: formula_train.csv       
  inflating: test.csv                
  inflating: train.csv               


# Анализ датасета

Сразу видим очень много признаков. Все признаки численные, пропуски отсутствуют. Из-за кол-ва признаков скорее всего придется снижать размерность пространства признаков

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV

plt.style.use("ggplot")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

df = pd.read_csv("/content/train.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17010 entries, 0 to 17009
Data columns (total 82 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   number_of_elements               17010 non-null  int64  
 1   mean_atomic_mass                 17010 non-null  float64
 2   wtd_mean_atomic_mass             17010 non-null  float64
 3   gmean_atomic_mass                17010 non-null  float64
 4   wtd_gmean_atomic_mass            17010 non-null  float64
 5   entropy_atomic_mass              17010 non-null  float64
 6   wtd_entropy_atomic_mass          17010 non-null  float64
 7   range_atomic_mass                17010 non-null  float64
 8   wtd_range_atomic_mass            17010 non-null  float64
 9   std_atomic_mass                  17010 non-null  float64
 10  wtd_std_atomic_mass              17010 non-null  float64
 11  mean_fie                         17010 non-null  float64
 12  wtd_mean_fie      

In [ ]:
print(f"Total objects: {len(df)}")

print(f"Total missing values: {df.isna().iloc[:,0].sum()}")

print()

print(f"Number of   numerical columns: {df.dtypes[df.dtypes != "object"].value_counts().sum()}")
print(f"Number of categorical columns: {df.dtypes[df.dtypes == "object"].value_counts().sum()}")

Total objects: 17010
Total missing values: 0

Number of   numerical columns: 82
Number of categorical columns: 0


In [ ]:
df

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,4,88.944468,57.862692,66.361592,36.116612,1.181795,1.062396,122.90607,31.794921,51.968828,53.622535,775.425000,1010.268571,718.152900,938.016780,1.305967,0.791488,810.6,735.985714,323.811808,355.562967,160.250000,105.514286,136.126003,84.528423,1.259244,1.207040,205,42.914286,75.237540,69.235569,4654.35725,2961.502286,724.953211,53.543811,1.033129,0.814598,8958.571,1579.583429,3306.162897,3572.596624,81.837500,111.727143,60.123179,99.414682,1.159687,0.787382,127.05,80.987143,51.433712,42.558396,6.905500,3.846857,3.479475,1.040986,1.088575,0.994998,12.878,1.744571,4.599064,4.666920,107.756645,61.015189,7.062488,0.621979,0.308148,0.262848,399.97342,57.127669,168.854244,138.517163,2.25,2.257143,2.213364,2.219783,1.368922,1.066221,1,1.085714,0.433013,0.437059,29.00
1,5,92.729214,58.518416,73.132787,36.396602,1.449309,1.057755,122.90607,36.161939,47.094633,53.979870,766.440000,1010.612857,720.605511,938.745413,1.544145,0.807078,810.6,743.164286,290.183029,354.963511,161.200000,104.971429,141.465215,84.370167,1.508328,1.204115,205,50.571429,67.321319,68.008817,5821.48580,3021.016571,1237.095080,54.095718,1.314442,0.914802,10488.571,1667.383429,3767.403176,3632.649185,90.890000,112.316429,69.833315,101.166398,1.427997,0.838666,127.05,81.207857,49.438167,41.667621,7.784400,3.796857,4.403790,1.035251,1.374977,1.073094,12.878,1.595714,4.473363,4.603000,172.205316,61.372331,16.064228,0.619735,0.847404,0.567706,429.97342,51.413383,198.554600,139.630922,2.00,2.257143,1.888175,2.210679,1.557113,1.047221,2,1.128571,0.632456,0.468606,26.00
2,4,88.944468,57.885242,66.361592,36.122509,1.181795,0.975980,122.90607,35.741099,51.968828,53.656268,775.425000,1010.820000,718.152900,939.009036,1.305967,0.773620,810.6,743.164286,323.811808,354.804183,160.250000,104.685714,136.126003,84.214573,1.259244,1.132547,205,49.314286,75.237540,67.797712,4654.35725,2999.159429,724.953211,53.974022,1.033129,0.760305,8958.571,1667.383429,3306.162897,3592.019281,81.837500,112.213571,60.123179,101.082152,1.159687,0.786007,127.05,81.207857,51.433712,41.639878,6.905500,3.822571,3.479475,1.037439,1.088575,0.927479,12.878,1.757143,4.599064,4.649635,107.756645,60.943760,7.062488,0.619095,0.308148,0.250477,399.97342,57.127669,168.854244,138.540613,2.25,2.271429,2.213364,2.232679,1.368922,1.029175,1,1.114286,0.433013,0.444697,19.00
3,4,88.944468,57.873967,66.361592,36.119560,1.181795,1.022291,122.90607,33.

In [ ]:
df.head(10)

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
0,4,88.944468,57.862692,66.361592,36.116612,1.181795,1.062396,122.90607,31.794921,51.968828,53.622535,775.425,1010.268571,718.152900,938.016780,1.305967,0.791488,810.6,735.985714,323.811808,355.562967,160.25,105.514286,136.126003,84.528423,1.259244,1.207040,205,42.914286,75.237540,69.235569,4654.35725,2961.502286,724.953211,53.543811,1.033129,0.814598,8958.571,1579.583429,3306.162897,3572.596624,81.8375,111.727143,60.123179,99.414682,1.159687,0.787382,127.05,80.987143,51.433712,42.558396,6.9055,3.846857,3.479475,1.040986,1.088575,0.994998,12.878,1.744571,4.599064,4.666920,107.756645,61.015189,7.062488,0.621979,0.308148,0.262848,399.97342,57.127669,168.854244,138.517163,2.25,2.257143,2.213364,2.219783,1.368922,1.066221,1,1.085714,0.433013,0.437059,29.0
1,5,92.729214,58.518416,73.132787,36.396602,1.449309,1.057755,122.90607,36.161939,47.094633,53.979870,766.440,1010.612857,720.605511,938.745413,1.544145,0.807078,810.6,743.164286,290.183029,354.963511,161.20,104.971429,141.465215,84.370167,1.508328,1.204115,205,50.571429,67.321319,68.008817,5821.48580,3021.016571,1237.095080,54.095718,1.314442,0.914802,10488.571,1667.383429,3767.403176,3632.649185,90.8900,112.316429,69.833315,101.166398,1.427997,0.838666,127.05,81.207857,49.438167,41.667621,7.7844,3.796857,4.403790,1.035251,1.374977,1.073094,12.878,1.595714,4.473363,4.603000,172.205316,61.372331,16.064228,0.619735,0.847404,0.567706,429.97342,51.413383,198.554600,139.630922,2.00,2.257143,1.888175,2.210679,1.557113,1.047221,2,1.128571,0.632456,0.468606,26.0
2,4,88.944468,57.885242,66.361592,36.122509,1.181795,0.975980,122.90607,35.741099,51.968828,53.656268,775.425,1010.820000,718.152900,939.009036,1.305967,0.773620,810.6,743.164286,323.811808,354.804183,160.25,104.685714,136.126003,84.214573,1.259244,1.132547,205,49.314286,75.237540,67.797712,4654.35725,2999.159429,724.953211,53.974022,1.033129,0.760305,8958.571,1667.383429,3306.162897,3592.019281,81.8375,112.213571,60.123179,101.082152,1.159687,0.786007,127.05,81.207857,51.433712,41.639878,6.9055,3.822571,3.479475,1.037439,1.088575,0.927479,12.878,1.757143,4.599064,4.649635,107.756645,60.943760,7.062488,0.619095,0.308148,0.250477,399.97342,57.127669,168.854244,138.540613,2.25,2.271429,2.213364,2.232679,1.368922,1.029175,1,1.114286,0.433013,0.444697,19.0
3,4,88.944468,57.873967,66.361592,36.119560,1.181795,1.022291,122.90607,33.768010,51.968828,53.639405,775.425,1

In [ ]:
df.sample(10)

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
13045,2,97.905940,95.406160,97.778206,95.311170,0.691843,0.582911,9.999120,43.953410,4.999560,4.329746,691.550000,677.575000,690.984949,677.154054,0.692330,0.578700,55.9,317.825000,27.950000,24.205410,185.500000,191.750000,185.078362,191.430185,0.690875,0.533848,25,105.250000,12.500000,10.825318,10510.000000,9540.000000,10329.399789,9408.663890,0.676013,0.631497,3880.000,3315.000000,1940.000000,1680.089283,99.400000,93.500000,98.697112,92.983155,0.686084,0.608582,23.60,37.900000,11.800000,10.219100,24.250000,25.525000,24.115555,25.422370,0.687608,0.517298,5.100,14.675000,2.550000,2.208365,102.000000,78.000000,90.000000,69.713700,0.577922,0.692407,96.00000,3.000000,48.000000,41.569219,5.500000,5.250000,5.477226,5.233176,0.689009,0.598270,1,2.250000,0.500000,0.433013,2.797
14806,2,141.720500,124.888333,132.419045,116.950838,0.628252,0.692807,100.993000,3.256333,50.496500,47.608557,763.900000,729.166667,756.759916,722.915564,0.683815,0.671712,208.4,150.433333,104.200000,98.240702,193.000000,197.333333,192.561678,196.940778,0.690877,0.614260,26,77.333333,13.000000,12.256518,14535.500000,11860.666667,12119.742572,9852.463365,0.531886,0.656777,16049.000,3179.333333,8024.500000,7565.571154,97.550000,79.233333,80.600868,65.167293,0.524811,0.652512,109.90,22.433333,54.950000,51.807357,23.500000,22.666667,23.366643,22.549521,0.687478,0.665204,5.000,5.333333,2.500000,2.357023,86.500000,65.333333,58.736701,42.971678,0.391953,0.544886,127.00000,34.666667,63.500000,59.868374,5.000000,4.666667,4.898979,4.578857,0.673012,0.682908,2,0.666667,1.000000,0.942809,7.500
5058,4,88.944468,57.871712,66.361592,36.118971,1.181795,1.030748,122.906070,33.373392,51.968828,53.636031,775.425000,1010.489143,718.152900,938.413556,1.305967,0.784955,810.6,738.857143,323.811808,355.259750,160.250000,105.182857,136.126003,84.402743,1.259244,1.180289,205,45.474286,75.237540,68.665240,4654.357250,2976.565143,724.953211,53.715482,1.033129,0.794232,8958.571,1614.703429,3306.162897,3580.425858,81.837500,111.921714,60.123179,100.078344,1.159687,0.787026,127.05,81.075429,51.433712,42.194061,6.905500,3.837143,3.479475,1.039566,1.088575,0.970617,12.878,1.744571,4.599064,4.660029,107.756645,60.986617,7.062488,0.620824,0.308148,0.258259,399.97342,57.127669,168.854244,138.526547,2.250000,2.262857,2.213364,2.224933,1.368922,1.052472,1,1.097143,0.433013,0.440185,26.100
10102,2,94.612000,97.032000,94.551319,97.0

In [ ]:
df.describe()

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
count,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000,17010.000000
mean,4.113874,87.534919,73.000381,71.308789,58.599393,1.165500,1.063972,115.443468,33.228642,44.320187,41.347802,769.945102,870.933167,737.791166,833.187164,1.298859,0.925990,572.564891,484.011001,215.783192,224.335460,157.931593,134.658921,144.381822,120.903678,1.267346,1.130936,139.402293,51.369816,51.628650,52.381817,6108.635376,5265.400936,3460.446536,3120.293657,1.072023,0.856086,8658.461969,2900.952855,3412.752366,3312.148043,77.003663,92.830366,54.420073,72.480817,1.069598,0.770660,120.962755,59.372269,48.985877,44.469852,14.325644,13.859314,10.169378,10.151939,1.093428,0.914854,21.132295,8.200406,8.317290,7.717151,89.876640,81.592442,29.851146,27.324396,0.727474,0.540722,251.268731,62.041541,99.092786,96.275334,3.197028,3.151430,3.055206,3.054358,1.295344,1.052631,2.042034,1.482988,0.839731,0.673590,34.502993
std,1.437846,29.786319,33.730910,31.166777,36.902657,0.364607,0.401877,54.614167,27.104426,19.993413,19.930640,87.402203,143.051039,78.290392,119.510733,0.382058,0.333773,309.303523,223.827363,109.825004,127.914578,20.105135,28.762215,22.047738,35.809683,0.375528,0.407911,67.264931,35.081827,22.905399,25.319727,2855.693051,3242.812464,3712.571509,3995.713727,0.341475,0.319447,4088.612125,2417.426098,1665.215317,1603.135763,27.748074,32.309955,29.158061,31.701324,0.342165,0.284759,58.926401,28.668

In [ ]:
float64_cols = df.select_dtypes(include=['float64']).columns
int64_cols = df.select_dtypes(include=['int64']).columns

# df[float64_cols] = df[float64_cols].astype(np.float32)
# df[int64_cols] = df[int64_cols].astype(np.int32)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17010 entries, 0 to 17009
Data columns (total 82 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   number_of_elements               17010 non-null  int64  
 1   mean_atomic_mass                 17010 non-null  float64
 2   wtd_mean_atomic_mass             17010 non-null  float64
 3   gmean_atomic_mass                17010 non-null  float64
 4   wtd_gmean_atomic_mass            17010 non-null  float64
 5   entropy_atomic_mass              17010 non-null  float64
 6   wtd_entropy_atomic_mass          17010 non-null  float64
 7   range_atomic_mass                17010 non-null  float64
 8   wtd_range_atomic_mass            17010 non-null  float64
 9   std_atomic_mass                  17010 non-null  float64
 10  wtd_std_atomic_mass              17010 non-null  float64
 11  mean_fie                         17010 non-null  float64
 12  wtd_mean_fie      

In [ ]:
def get_num_outliers(column):
    q1 = np.percentile(column, 25)
    q3 = np.percentile(column, 75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return ((column < lower) | (column > upper)).sum()


print("Number of outliers for each column:")

max_col_width = max(len(col) for col in df.columns)

for column in df.columns:
    print(f"{column:<{max_col_width}}: {get_num_outliers(df[column])}")

Number of outliers for each column:
number_of_elements             : 11
mean_atomic_mass               : 1285
wtd_mean_atomic_mass           : 1028
gmean_atomic_mass              : 2668
wtd_gmean_atomic_mass          : 1022
entropy_atomic_mass            : 314
wtd_entropy_atomic_mass        : 0
range_atomic_mass              : 0
wtd_range_atomic_mass          : 1315
std_atomic_mass                : 3
wtd_std_atomic_mass            : 7
mean_fie                       : 1552
wtd_mean_fie                   : 0
gmean_fie                      : 1194
wtd_gmean_fie                  : 2
entropy_fie                    : 235
wtd_entropy_fie                : 1764
range_fie                      : 0
wtd_range_fie                  : 0
std_fie                        : 0
wtd_std_fie                    : 0
mean_atomic_radius             : 967
wtd_mean_atomic_radius         : 10
gmean_atomic_radius            : 1205
wtd_gmean_atomic_radius        : 7
entropy_atomic_radius          : 235
wtd_entropy_atomi

In [ ]:
corr_matrix = df.corr()
upper_triangle = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

high_corr = corr_matrix.where(upper_triangle) > 0.9

num_high_corr_pairs = high_corr.sum().sum()

print(f"Число пар признаков с корреляцией > 0.9: {num_high_corr_pairs}")

Число пар признаков с корреляцией > 0.9: 73


In [ ]:
df.corr().where(lambda x: x > 0.9)

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
number_of_elements,1.000000,NaN,NaN,NaN,NaN,0.939359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.972867,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.971904,0.903625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.900813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.967576,NaN,NaN,NaN,NaN,NaN,NaN
mean_atomic_mass,NaN,1.000000,NaN,0.941275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wtd_mean_atomic_mass,NaN,NaN,1.000000,NaN,0.965004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gmean_atomic_mass,NaN,0.941275,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wtd_gmean_atomic_mass,NaN,NaN,0.965004,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
entropy_atomic_mass,0.939359,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.964976,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.972664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.931501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.928157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.963567,NaN,NaN,NaN,NaN,NaN,NaN
wtd_entropy_atomic_mass,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.962112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [ ]:
df.columns

Index(['number_of_elements', 'mean_atomic_mass', 'wtd_mean_atomic_mass',
       'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass',
       'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass',
       'std_atomic_mass', 'wtd_std_atomic_mass', 'mean_fie', 'wtd_mean_fie',
       'gmean_fie', 'wtd_gmean_fie', 'entropy_fie', 'wtd_entropy_fie',
       'range_fie', 'wtd_range_fie', 'std_fie', 'wtd_std_fie',
       'mean_atomic_radius', 'wtd_mean_atomic_radius', 'gmean_atomic_radius',
       'wtd_gmean_atomic_radius', 'entropy_atomic_radius',
       'wtd_entropy_atomic_radius', 'range_atomic_radius',
       'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius',
       'mean_Density', 'wtd_mean_Density', 'gmean_Density',
       'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density',
       'range_Density', 'wtd_range_Density', 'std_Density', 'wtd_std_Density',
       'mean_ElectronAffinity', 'wtd_mean_ElectronAffinity',
       'gmean_

In [ ]:
for column in df.columns:
    if "wtd" in column:
        print(column)

wtd_mean_atomic_mass
wtd_gmean_atomic_mass
wtd_entropy_atomic_mass
wtd_range_atomic_mass
wtd_std_atomic_mass
wtd_mean_fie
wtd_gmean_fie
wtd_entropy_fie
wtd_range_fie
wtd_std_fie
wtd_mean_atomic_radius
wtd_gmean_atomic_radius
wtd_entropy_atomic_radius
wtd_range_atomic_radius
wtd_std_atomic_radius
wtd_mean_Density
wtd_gmean_Density
wtd_entropy_Density
wtd_range_Density
wtd_std_Density
wtd_mean_ElectronAffinity
wtd_gmean_ElectronAffinity
wtd_entropy_ElectronAffinity
wtd_range_ElectronAffinity
wtd_std_ElectronAffinity
wtd_mean_FusionHeat
wtd_gmean_FusionHeat
wtd_entropy_FusionHeat
wtd_range_FusionHeat
wtd_std_FusionHeat
wtd_mean_ThermalConductivity
wtd_gmean_ThermalConductivity
wtd_entropy_ThermalConductivity
wtd_range_ThermalConductivity
wtd_std_ThermalConductivity
wtd_mean_Valence
wtd_gmean_Valence
wtd_entropy_Valence
wtd_range_Valence
wtd_std_Valence


In [ ]:
train_columns = list(df.columns)

# train_columns = [
#     "number_of_elements",
#     "wtd_mean_atomic_mass",
#     # "wtd_gmean_atomic_mass",
#     "wtd_entropy_atomic_mass",
#     "wtd_range_atomic_mass",
#     "wtd_std_atomic_mass",
#     "wtd_mean_fie",
#     # "wtd_gmean_fie",
#     # "wtd_entropy_fie",
#     "wtd_range_fie",
#     # "wtd_std_fie",
#     "wtd_mean_atomic_radius",
#     # "wtd_gmean_atomic_radius",
#     # "wtd_entropy_atomic_radius",
#     "wtd_range_atomic_radius",
#     "wtd_std_atomic_radius",
#     "wtd_mean_Density",
#     # "wtd_gmean_Density",
#     # "wtd_entropy_Density",
#     "wtd_range_Density",
#     "wtd_std_Density",
#     "wtd_mean_ElectronAffinity",
#     # "wtd_gmean_ElectronAffinity",
#     # "wtd_entropy_ElectronAffinity",
#     "wtd_range_ElectronAffinity",
#     "wtd_std_ElectronAffinity",
#     "wtd_mean_FusionHeat",
#     # "wtd_gmean_FusionHeat",
#     # "wtd_entropy_FusionHeat",
#     "wtd_range_FusionHeat",
#     "wtd_std_FusionHeat",
#     "wtd_mean_ThermalConductivity",
#     # "wtd_gmean_ThermalConductivity",
#     # "wtd_entropy_ThermalConductivity",
#     "wtd_range_ThermalConductivity",
#     "wtd_std_ThermalConductivity",
#     "wtd_mean_Valence",
#     # "wtd_gmean_Valence",
#     # "wtd_entropy_Valence",
#     "wtd_range_Valence",
#     "wtd_std_Valence",
#     "critical_temp",
# ]

# optimal_columns = [
#     "number_of_elements",
#     "wtd_mean_atomic_mass",
#     "wtd_mean_fie",
#     "wtd_mean_atomic_radius",
#     "wtd_mean_Density",
#     "wtd_mean_ElectronAffinity",
#     "wtd_mean_FusionHeat",
#     "wtd_mean_ThermalConductivity",
#     "wtd_mean_Valence",
#     "wtd_std_Valence",
#     "wtd_std_atomic_radius",
#     "critical_temp"
# ]

test_columns = train_columns.copy()
test_columns.remove("critical_temp")

df = df[train_columns].copy()

In [ ]:
corr_matrix = df.corr()
upper_triangle = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

high_corr = corr_matrix.where(upper_triangle) > 0.9

num_high_corr_pairs = high_corr.sum().sum()

print(f"Число пар признаков с корреляцией > 0.9: {num_high_corr_pairs}")

Число пар признаков с корреляцией > 0.9: 73


In [ ]:
df.corr().where(lambda x: x > 0.9)

,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,wtd_std_atomic_mass,mean_fie,wtd_mean_fie,gmean_fie,wtd_gmean_fie,entropy_fie,wtd_entropy_fie,range_fie,wtd_range_fie,std_fie,wtd_std_fie,mean_atomic_radius,wtd_mean_atomic_radius,gmean_atomic_radius,wtd_gmean_atomic_radius,entropy_atomic_radius,wtd_entropy_atomic_radius,range_atomic_radius,wtd_range_atomic_radius,std_atomic_radius,wtd_std_atomic_radius,mean_Density,wtd_mean_Density,gmean_Density,wtd_gmean_Density,entropy_Density,wtd_entropy_Density,range_Density,wtd_range_Density,std_Density,wtd_std_Density,mean_ElectronAffinity,wtd_mean_ElectronAffinity,gmean_ElectronAffinity,wtd_gmean_ElectronAffinity,entropy_ElectronAffinity,wtd_entropy_ElectronAffinity,range_ElectronAffinity,wtd_range_ElectronAffinity,std_ElectronAffinity,wtd_std_ElectronAffinity,mean_FusionHeat,wtd_mean_FusionHeat,gmean_FusionHeat,wtd_gmean_FusionHeat,entropy_FusionHeat,wtd_entropy_FusionHeat,range_FusionHeat,wtd_range_FusionHeat,std_FusionHeat,wtd_std_FusionHeat,mean_ThermalConductivity,wtd_mean_ThermalConductivity,gmean_ThermalConductivity,wtd_gmean_ThermalConductivity,entropy_ThermalConductivity,wtd_entropy_ThermalConductivity,range_ThermalConductivity,wtd_range_ThermalConductivity,std_ThermalConductivity,wtd_std_ThermalConductivity,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence,critical_temp
number_of_elements,1.000000,NaN,NaN,NaN,NaN,0.939359,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.972867,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.971904,0.903625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.900813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.967576,NaN,NaN,NaN,NaN,NaN,NaN
mean_atomic_mass,NaN,1.000000,NaN,0.941275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wtd_mean_atomic_mass,NaN,NaN,1.000000,NaN,0.965004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gmean_atomic_mass,NaN,0.941275,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wtd_gmean_atomic_mass,NaN,NaN,0.965004,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
entropy_atomic_mass,0.939359,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.964976,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.972664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.931501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.928157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.963567,NaN,NaN,NaN,NaN,NaN,NaN
wtd_entropy_atomic_mass,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.962112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [ ]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]

# Baseline — решение

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=42, test_size=0.2, shuffle=True)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)

print(f"{"R2":<4}:", model.score(X_valid, y_valid))
print(f"{"RMSE":<4}:", np.sqrt(mean_squared_error(y_valid, y_pred)))

R2  : 0.7263736035742483
RMSE: 17.644906836695498


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = model.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": y_pred
})

results.to_csv("sumbission_normal.csv", index=False)

In [ ]:
y_log_train = np.log1p(y_train)
y_log_valid = np.log1p(y_valid)

model = LinearRegression()
model.fit(X_train, y_log_train)
y_log_pred = model.predict(X_valid)

y_pred_original = np.expm1(y_log_pred)
y_valid_original = np.expm1(y_log_valid)

r2_real = r2_score(y_valid_original, y_pred_original)
rmse_real = np.sqrt(mean_squared_error(y_valid_original, y_pred_original))

print(f"{'R2':<4}: {r2_real:.4f}")
print(f"{'RMSE':<4}: {rmse_real:.4f}")

R2  : 0.7293
RMSE: 17.5510


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = model.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": np.expm1(y_pred)
})

results.to_csv("sumbission_normal_log.csv", index=False)

# Создание моделей

Большое число скоррелированных признаков подсказывает попробовать воспользоваться регуляризацией. Попробуем как L2, так и L1

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(
        alphas=np.logspace(-3, 1, 100),
        cv=5,
        scoring="neg_mean_squared_error"
    ))
])

pipe.fit(X_train, y_train)

best_alpha = pipe.named_steps["model"].alpha_

y_pred = pipe.predict(X_valid)

r2_real = r2_score(y_valid, y_pred)
rmse_real = np.sqrt(mean_squared_error(y_valid, y_pred))

print("Best alpha:", best_alpha)
print(f"{'R2':<4}: {r2_real:.4f}")
print(f"{'RMSE':<4}: {rmse_real:.4f}")

Best alpha: 0.05462277217684343
R2  : 0.7264
RMSE: 17.6447


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = pipe.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": y_pred
})

results.to_csv("submission_ridge.csv", index=False)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(
        alphas=np.logspace(-3, 1, 100),
        cv=5,
        scoring="neg_mean_squared_error"
    ))
])

y_log_train = np.log1p(y_train)
y_log_valid = np.log1p(y_valid)

pipe.fit(X_train, y_log_train)

best_alpha = pipe.named_steps["model"].alpha_

y_log_pred = pipe.predict(X_valid)

y_pred_original = np.expm1(y_log_pred)
y_valid_original = np.expm1(y_log_valid)

r2_real = r2_score(y_valid_original, y_pred_original)
rmse_real = np.sqrt(mean_squared_error(y_valid_original, y_pred_original))

print("Best alpha:", best_alpha)
print(f"{'R2':<4}: {r2_real:.4f}")
print(f"{'RMSE':<4}: {rmse_real:.4f}")

Best alpha: 0.10476157527896651
R2  : 0.7295
RMSE: 17.5451


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = pipe.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": np.expm1(y_pred)
})

results.to_csv("submission_ridge_log.csv", index=False)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(
        alphas=np.logspace(-5, -4, 100),
        # alphas=[2.1544346900318823e-05],
        cv=5,
        max_iter=30_000,
        tol=1e-4,
        random_state=42
    ))
])

pipe.fit(X_train, y_train)

best_alpha = pipe.named_steps["model"].alpha_
y_pred = pipe.predict(X_valid)

r2 = r2_score(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))

print("Best alpha:", best_alpha)
print(f"{'R2':<4}: {r2:.4f}")
print(f"{'RMSE':<4}: {rmse:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 536118.1002090359, tolerance: 1288.4169540315613
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 719818.1356532942, tolerance: 1296.9637487269472
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 12266.279038819019, tolerance: 1296.9637487269472
  model = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:681: Convergenc

Best alpha: 0.0001
R2  : 0.7263
RMSE: 17.6471


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.997e+05, tolerance: 1.618e+03
  model = cd_fast.enet_coordinate_descent(


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = pipe.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": y_pred
})

results.to_csv("submission_lasso.csv", index=False)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(
        alphas=np.logspace(-5, -3, 100),
        cv=5,
        max_iter=30_000,
        random_state=42
    ))
])

y_log_train = np.log1p(y_train)
y_log_valid = np.log1p(y_valid)

pipe.fit(X_train, y_log_train)

best_alpha = pipe.named_steps["model"].alpha_

y_log_pred = pipe.predict(X_valid)

y_pred_original = np.expm1(y_log_pred)
y_valid_original = np.expm1(y_log_valid)

r2_real = r2_score(y_valid_original, y_pred_original)
rmse_real = np.sqrt(mean_squared_error(y_valid_original, y_pred_original))

print("Best alpha:", best_alpha)
print(f"{'R2':<4}: {r2_real:.4f}")
print(f"{'RMSE':<4}: {rmse_real:.4f}")

Best alpha: 2.0092330025650458e-05
R2  : 0.7295
RMSE: 17.5432


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.977e+02, tolerance: 2.267e+00
  model = cd_fast.enet_coordinate_descent(


In [ ]:
df_test = pd.read_csv("/content/test.csv")[test_columns]

X_test = df_test.iloc[:, :]

y_pred = pipe.predict(X_test)

results = pd.DataFrame({
    "index": range(len(y_pred)),
    "critical_temp": np.expm1(y_pred)
})

results.to_csv("submission_lasso_log.csv", index=False)